# Entregable I: descripción del problema y análisis exploratorio

**Predicción de crecimiento de adopción en protocolos blockchain a partir de fundamentales *on-chain***
Modelos y Simulación de Sistemas II (Aprendizaje Supervisado), Universidad de Antioquia.
Michael Stiven Tabares Tobón, María Fernanda Atencia Oliva, Valeria Vásquez Barragán y Jhon Sebastián Martínez Soto.

El notebook sigue las secciones del reporte: **1. Descripción del problema** (A. contexto, B. composición de la base de datos, C. paradigma de aprendizaje) y **2. Estado del arte**. El código está en la parte B, que descarga los datos de la API pública de DeFiLlama, arma el panel semanal (protocolo, semana) descrito en el reporte, calcula las 24 variables y la variable objetivo, y genera:

* las cuatro figuras del reporte (`figs/*.pdf`);
* el archivo `resultados_eda.tex`, con las cifras que llenan los espacios en rojo de `main.tex`;
* `entrega_overleaf.zip`, que agrupa los dos puntos anteriores para subirlos a Overleaf;
* `datos/panel_eda.csv.gz` y `datos/dataset_modelo.csv.gz` (panel sin imputar y dataset listo para el Entregable II).

## Cómo usarlo

1. Abrir [Google Colab](https://colab.research.google.com) → *Archivo* → *Subir notebook* → elegir este archivo.
2. *Entorno de ejecución* → *Ejecutar todas*.
3. Esperar. La primera vez descarga el historial de TVL de todos los protocolos y puede tardar entre 20 y 60 minutos. Si una celda se interrumpe, vuelva a ejecutarla: lo ya descargado queda guardado en `datos/`. Si Colab se desconecta del todo, esos archivos se pierden; para conservarlos, ponga `USAR_DRIVE = True` antes de empezar.
4. Al final se descarga `entrega_overleaf.zip`. Descomprímalo en su computador: trae `resultados_eda.tex` y cuatro figuras en PDF.
5. En Overleaf, use *Upload* para subir esos cinco archivos al proyecto, en la misma carpeta de `main.tex` (las figuras también pueden ir en una carpeta `figs/`).
6. Recompilar: las cifras y las figuras se llenan solas en el documento.

Si el tiempo apremia, cambie `MODO_RAPIDO = True` en la celda de parámetros: solo descarga los protocolos con TVL actual ≥ 1 M USD (más rápido, pero deja fuera los que ya desaparecieron).

## 1. Descripción del problema

### A. Contexto del problema y utilidad de una solución basada en ML
Las finanzas descentralizadas (DeFi) ofrecen servicios financieros mediante contratos inteligentes, sin intermediarios. Su métrica más usada es el valor total bloqueado (*Total Value Locked*, TVL): los activos depositados en los contratos de cada protocolo. DeFiLlama lista miles de protocolos (la celda B.2 imprime cuántos) y el TVL agregado pasa por ciclos abruptos de crecimiento y contracción, de modo que seguir a mano cuáles crecen es inviable.

**Pregunta de investigación.** ¿La información fundamental y la historia reciente de un protocolo, disponibles hasta la semana *t*, permiten anticipar si estará en el 10 % de protocolos con mayor crecimiento relativo del TVL en la semana *t+1*?

Un modelo supervisado permite procesar el panel completo de cada semana (más de mil protocolos), capturar relaciones no lineales entre fundamentales y crecimiento, y evaluar de forma verificable si existe señal predictiva (Sec. I-A del reporte).

### B. Composición de la base de datos
Las celdas B.0 a B.9 construyen la base de datos de la Sec. I-B del reporte: descarga desde la API pública de DeFiLlama, panel semanal (protocolo, semana), universo elegible, las 24 variables predictoras y la variable objetivo, datos faltantes, imputación, codificación y análisis exploratorio (Tabla II y Figs. 1 y 2).

#### B.0 Parámetros
Coinciden con las definiciones del reporte (Sec. I-B).

In [ ]:
FECHA_INICIO = "2022-01-03"   # lunes de la primera semana ISO del período de estudio
TVL_MIN = 1_000_000           # TVL mínimo (USD) para que un protocolo sea elegible en una semana
HIST_MIN = 8                  # semanas de historial previo requeridas
RELLENO_MAX_DIAS = 7          # relleno hacia adelante máximo en la serie diaria de TVL
VENTANAS_G = [1, 2, 4, 8, 12, 26]     # crecimiento del TVL (semanas)
VENTANAS_MOM = [2, 4, 12, 26]         # momentum (semanas)
VENTANAS_VOL = [4, 8, 12, 26]         # volatilidad (semanas)
MIN_PROTOS_CATEGORIA = 20     # categorías con menos protocolos se agrupan en "Otros"

MODO_RAPIDO = False   # True: solo protocolos con TVL actual >= TVL_MIN (más rápido, con sesgo de supervivencia)
HILOS = 4             # descargas en paralelo; bájelo a 2 si la API responde con muchos errores o falta memoria
USAR_DRIVE = False    # True: guarda los datos en Google Drive para no descargarlos de nuevo

API = "https://api.llama.fi"
CARPETA_DATOS = "datos"
CARPETA_FIGS = "figs"

#### B.1 Librerías y cliente de la API
Cada descarga se guarda en `datos/` (un archivo por protocolo). Si una celda se interrumpe, al volver a ejecutarla solo se descarga lo que falta.

In [ ]:
import os, re, json, time, random, zipfile, datetime as dt
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap

try:
    from tqdm.auto import tqdm
except ImportError:          # sin tqdm: se imprime un avance simple
    def tqdm(iterable, total=None, desc=""):
        for i, x in enumerate(iterable, 1):
            if total and (i % max(1, total // 10) == 0 or i == total):
                print(f"{desc}: {i}/{total}")
            yield x

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DATOS = "/content/drive/MyDrive/defillama_datos"

for sub in ("", "tvl", "fees", "dexs"):
    os.makedirs(os.path.join(CARPETA_DATOS, sub), exist_ok=True)
os.makedirs(CARPETA_FIGS, exist_ok=True)

SESION = requests.Session()
SESION.headers.update({"User-Agent": "UdeA-ModelosII-proyecto-DeFi"})


def get_json(ruta, params=None, reintentos=5, reintentos_5xx=2):
    """GET a la API de DeFiLlama con reintentos.
    Devuelve (estado, datos), con estado 'ok', 'no_existe' (400/404) o 'error'."""
    errores_5xx = 0
    for intento in range(reintentos):
        try:
            r = SESION.get(API + ruta, params=params, timeout=120)
        except requests.RequestException:
            time.sleep(min(60, 2 ** intento) + random.random())
            continue
        if r.status_code == 200:
            try:
                return "ok", r.json()
            except ValueError:
                return "error", None
        if r.status_code in (400, 404):
            return "no_existe", None
        if r.status_code >= 500:
            errores_5xx += 1
            if errores_5xx > reintentos_5xx:
                return "error", None
        time.sleep(min(60, 2 ** intento) + random.random())   # 429 o 5xx: esperar y reintentar
    return "error", None


def a_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


def nombre_archivo(slug):
    return re.sub(r"[^A-Za-z0-9._-]", "_", str(slug))


def ruta_cache(tipo, slug, ext):
    return os.path.join(CARPETA_DATOS, tipo, nombre_archivo(slug) + ext)


def en_cache(tipo, slug):
    """True si ya hay un resultado definitivo (datos o 'no existe') para ese protocolo."""
    return os.path.exists(ruta_cache(tipo, slug, ".csv")) or os.path.exists(ruta_cache(tipo, slug, ".none"))


def guardar_cache(tipo, slug, estado, df):
    for ext in (".csv", ".none", ".err"):
        if os.path.exists(ruta_cache(tipo, slug, ext)):
            os.remove(ruta_cache(tipo, slug, ext))
    if estado == "ok" and df is not None and len(df):
        df.to_csv(ruta_cache(tipo, slug, ".csv"), index=False)
    elif estado in ("ok", "no_existe"):
        open(ruta_cache(tipo, slug, ".none"), "w").close()
    else:
        open(ruta_cache(tipo, slug, ".err"), "w").close()   # se reintenta al volver a ejecutar


def descargar_en_paralelo(funcion, slugs, desc):
    conteo = {}
    with ThreadPoolExecutor(max_workers=HILOS) as ex:
        futuros = [ex.submit(funcion, s) for s in slugs]
        for f in tqdm(as_completed(futuros), total=len(futuros), desc=desc):
            estado = f.result()
            conteo[estado] = conteo.get(estado, 0) + 1
    return conteo


def a_fechas(ts):
    """Marcas de tiempo Unix (segundos) -> fechas UTC sin hora."""
    t = pd.to_datetime(pd.to_numeric(pd.Series(ts), errors="coerce"), unit="s", utc=True)
    return pd.DatetimeIndex(t).tz_convert(None).normalize()

#### B.2 Lista de protocolos
`GET /protocols`. Se excluyen los exchanges centralizados (categoría *CEX*), que no son protocolos DeFi. La fecha de descarga queda guardada en `datos/meta.json` y define la última semana completa del período.

In [ ]:
ruta_meta = os.path.join(CARPETA_DATOS, "meta.json")
ruta_lista = os.path.join(CARPETA_DATOS, "protocols.json")
if os.path.exists(ruta_lista) and os.path.exists(ruta_meta):
    with open(ruta_lista, encoding="utf-8") as f:
        lista = json.load(f)
    with open(ruta_meta, encoding="utf-8") as f:
        meta = json.load(f)
    print("Usando la lista de protocolos descargada el", meta["fecha_descarga"])
else:
    estado, lista = get_json("/protocols", reintentos=8, reintentos_5xx=5)
    if estado != "ok" or not isinstance(lista, list):
        raise RuntimeError("No se pudo descargar /protocols. Revise la conexión y vuelva a ejecutar esta celda.")
    meta = {"fecha_descarga": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d")}
    with open(ruta_lista, "w", encoding="utf-8") as f:
        json.dump(lista, f)
    with open(ruta_meta, "w", encoding="utf-8") as f:
        json.dump(meta, f)

FECHA_DESCARGA = pd.Timestamp(meta["fecha_descarga"])
# Última semana ISO completa: la que cierra el domingo anterior a la fecha de descarga
ULTIMA_SEMANA = FECHA_DESCARGA - pd.Timedelta(days=FECHA_DESCARGA.dayofweek + 1)
PRIMERA_SEMANA = pd.Timestamp(FECHA_INICIO) + pd.Timedelta(days=6)   # domingo que cierra la primera semana


def slug_de(p):
    s = p.get("slug")
    if not s:
        s = re.sub(r"[^a-z0-9]+", "-", str(p.get("name", "")).lower()).strip("-")
    return str(s)


protos = pd.DataFrame([{
    "slug": slug_de(p),
    "id": str(p.get("id", "")),
    "nombre": p.get("name"),
    "categoria": p.get("category") or "Sin categoría",
    "tvl_actual": a_float(p.get("tvl")),
} for p in lista if isinstance(p, dict)]).drop_duplicates("slug")

n_listados = len(protos)
protos = protos[protos["categoria"] != "CEX"]
if MODO_RAPIDO:
    protos = protos[protos["tvl_actual"] >= TVL_MIN]
protos = protos.reset_index(drop=True)
info = protos.set_index("slug")

print(f"Protocolos listados: {n_listados} | a descargar: {len(protos)}")
print("Fecha de descarga:", FECHA_DESCARGA.date(), "| última semana completa:", ULTIMA_SEMANA.date())

#### B.3 Historial de TVL
`GET /protocol/{slug}` (campo `tvl`) para cada protocolo y `GET /v2/historicalChainTvl` para el TVL agregado. Es el paso más lento.

In [ ]:
def bajar_tvl(slug):
    if en_cache("tvl", slug):
        return "en caché"
    estado, js = get_json(f"/protocol/{slug}")
    df = None
    if estado == "ok" and isinstance(js, dict):
        filas = [(p.get("date"), p.get("totalLiquidityUSD")) for p in (js.get("tvl") or []) if isinstance(p, dict)]
        df = pd.DataFrame(filas, columns=["ts", "tvl"]) if filas else None
    guardar_cache("tvl", slug, estado, df)
    return estado


t0 = time.time()
print(descargar_en_paralelo(bajar_tvl, protos["slug"].tolist(), "TVL por protocolo"), f"| {time.time() - t0:.0f} s")
errores = [s for s in protos["slug"] if os.path.exists(ruta_cache("tvl", s, ".err"))]
if errores:
    print(f"Atención: {len(errores)} protocolos con error de descarga. Vuelva a ejecutar esta celda para reintentarlos.")

estado, agregado = get_json("/v2/historicalChainTvl", reintentos=8, reintentos_5xx=5)
if estado == "ok" and agregado:
    pd.DataFrame(agregado).to_csv(os.path.join(CARPETA_DATOS, "tvl_agregado.csv"), index=False)

#### B.4 Panel semanal de TVL y universo elegible
Serie diaria → relleno hacia adelante de hasta 7 días (nunca hacia atrás) → TVL al cierre de cada semana ISO (domingo, UTC).
Un protocolo es elegible en la semana *t* si su TVL es ≥ 1 M USD y tiene al menos 8 semanas de historial previo.

In [ ]:
def tvl_semanal(df):
    s = pd.Series(pd.to_numeric(df["tvl"], errors="coerce").to_numpy(), index=a_fechas(df["ts"]))
    s = s[s.index.notna() & (s.index <= ULTIMA_SEMANA)]
    s = s.groupby(level=0).last()                      # un dato por día
    s = s.where(s >= 0).dropna()
    if s.empty:
        return None
    s = s.asfreq("D").ffill(limit=RELLENO_MAX_DIAS)    # huecos cortos: relleno hacia adelante
    return s.resample("W-SUN").last()                  # valor al cierre de la semana


series = {}
for slug in tqdm(protos["slug"].tolist(), total=len(protos), desc="Series semanales"):
    if os.path.exists(ruta_cache("tvl", slug, ".csv")):
        w = tvl_semanal(pd.read_csv(ruta_cache("tvl", slug, ".csv")))
        if w is not None and w.notna().any():
            series[slug] = w

TVL = pd.concat(series, axis=1, sort=True)
TVL = TVL.reindex(pd.date_range(TVL.index.min(), ULTIMA_SEMANA, freq="W-SUN"))

valido = TVL.gt(0)
hist = valido.astype(float).shift(1).fillna(0).cumsum()      # semanas con TVL antes de t
ELIG = TVL.ge(TVL_MIN) & hist.ge(HIST_MIN)
fuera = (TVL.index < PRIMERA_SEMANA) | (TVL.index > ULTIMA_SEMANA - pd.Timedelta(days=7))
ELIG.loc[fuera, :] = False                                   # t+1 debe existir para la etiqueta

alguna_vez = ELIG.any(axis=0)
TVL, ELIG = TVL.loc[:, alguna_vez], ELIG.loc[:, alguna_vez]
print(f"Protocolos con historial: {len(series)} | elegibles al menos una semana: {TVL.shape[1]}")

#### B.5 Fees y volumen DEX
`GET /summary/fees/{slug}` y `GET /summary/dexs/{slug}`, solo para los protocolos elegibles. Las listas `GET /overview/fees` y `GET /overview/dexs` indican cuáles tienen datos, para no consultar a los demás.

In [ ]:
elegibles = list(TVL.columns)


def ids_overview(ruta, data_type):
    estado, js = get_json(ruta, params={"excludeTotalDataChart": "true",
                                        "excludeTotalDataChartBreakdown": "true",
                                        "dataType": data_type}, reintentos=6, reintentos_5xx=3)
    if estado != "ok" or not isinstance(js, dict):
        return None
    slugs, ids = set(), set()
    for p in js.get("protocols") or []:
        if isinstance(p, dict):
            if p.get("slug"):
                slugs.add(str(p["slug"]))
            for k in ("defillamaId", "id"):
                if p.get(k) is not None:
                    ids.add(str(p[k]))
    return slugs, ids


def candidatos(overview):
    if overview is None:
        return elegibles                    # sin la lista se consulta a todos
    slugs, ids = overview
    elegidos = [s for s in elegibles if s in slugs or info.at[s, "id"] in ids]
    # Si casi nada coincide, la lista no es confiable y se consulta a todos
    return elegidos if len(elegidos) >= 0.02 * len(elegibles) else elegibles


def bajador(tipo, data_type):
    def bajar(slug):
        if en_cache(tipo, slug):
            return "en caché"
        estado, js = get_json(f"/summary/{tipo}/{slug}",
                              params={"excludeTotalDataChart": "false",
                                      "excludeTotalDataChartBreakdown": "true",
                                      "dataType": data_type},
                              reintentos=4, reintentos_5xx=1)
        df = None
        if estado == "ok" and isinstance(js, dict):
            filas = []
            for punto in js.get("totalDataChart") or []:
                if isinstance(punto, (list, tuple)) and len(punto) >= 2:
                    filas.append((punto[0], punto[1]))
                elif isinstance(punto, dict):
                    filas.append((punto.get("date"), punto.get("value")))
            df = pd.DataFrame(filas, columns=["ts", "valor"]) if filas else None
        guardar_cache(tipo, slug, estado, df)
        return estado
    return bajar


cand_fees = candidatos(ids_overview("/overview/fees", "dailyFees"))
cand_dexs = candidatos(ids_overview("/overview/dexs", "dailyVolume"))
print(f"Elegibles: {len(elegibles)} | a consultar fees: {len(cand_fees)} | a consultar volumen DEX: {len(cand_dexs)}")
print("Fees:", descargar_en_paralelo(bajador("fees", "dailyFees"), cand_fees, "Fees"))
print("Volumen DEX:", descargar_en_paralelo(bajador("dexs", "dailyVolume"), cand_dexs, "Volumen DEX"))


def suma_semanal(tipo, slug):
    p = ruta_cache(tipo, slug, ".csv")
    if not os.path.exists(p):
        return None
    df = pd.read_csv(p)
    s = pd.Series(pd.to_numeric(df["valor"], errors="coerce").to_numpy(), index=a_fechas(df["ts"]))
    s = s[s.index.notna() & (s.index <= ULTIMA_SEMANA)]
    s = s.where(s >= 0).dropna()
    if s.empty:
        return None
    return s.groupby(level=0).sum().resample("W-SUN").sum(min_count=1)   # suma de los 7 días


def matriz(tipo):
    datos = {s: suma_semanal(tipo, s) for s in elegibles}
    datos = {k: v for k, v in datos.items() if v is not None}
    M = pd.concat(datos, axis=1, sort=True) if datos else pd.DataFrame(index=TVL.index)
    return M.reindex(index=TVL.index, columns=TVL.columns)


FEES = matriz("fees")      # NaN = el protocolo no reporta fees esa semana
VOL = matriz("dexs")       # NaN = sin volumen DEX esa semana
print(f"Protocolos elegibles con fees: {FEES.notna().any().sum()} | con volumen DEX: {VOL.notna().any().sum()}")

#### B.6 Variables predictoras y variable objetivo
Las 24 variables de la Tabla I, calculadas con información hasta el cierre de la semana *t*, y la etiqueta de la ec. (2): *y* = 1 si el crecimiento logarítmico del TVL de *t* a *t+1* está en el decil superior de la semana. Si un protocolo deja de reportar TVL en *t+1*, *y* = 0.

In [ ]:
with np.errstate(divide="ignore", invalid="ignore"):
    LT = np.log1p(TVL)                                  # ln(1 + TVL)
    G = {k: LT - LT.shift(k) for k in VENTANAS_G}       # crecimiento a k semanas
    G1 = G[1]
    MOM = {k: G1.rolling(k, min_periods=k).mean() for k in VENTANAS_MOM}
    ACCEL = G1 - G1.shift(1)
    ACCEL12 = MOM[4] - MOM[12]
    VOLAT = {k: G1.rolling(k, min_periods=k).std() for k in VENTANAS_VOL}

    LF, LV = np.log1p(FEES), np.log1p(VOL)
    RF, RV = FEES / TVL, VOL / TVL

    SIG = TVL.shift(-1)
    G_OBJ = np.log(SIG.where(SIG > 0) / TVL)            # ln(TVL_{t+1} / TVL_t)

G_E = G_OBJ.where(ELIG)
Q90 = G_E.quantile(0.9, axis=1)                         # umbral del decil superior de cada semana
Y = G_E.ge(Q90, axis=0) & ELIG                          # sin TVL en t+1 -> y = 0

VARS = {
    "log_fees_week": LF,
    "g_fees_1w": LF - LF.shift(1),
    "g_fees_4w": LF - LF.shift(4),
    "log_dex_vol_week": LV,
    "ratio_fees_tvl": RF,
    "ratio_vol_tvl": RV,
    "fees_tvl_rank_w": RF.where(ELIG).rank(axis=1, pct=True),
    **{f"g_tvl_{k}w": G[k] for k in VENTANAS_G},
    **{f"mom_{k}w": MOM[k] for k in VENTANAS_MOM},
    "accel": ACCEL,
    "accel_12w": ACCEL12,
    **{f"vol_{k}w": VOLAT[k] for k in VENTANAS_VOL},
    "tvl_rank_w": TVL.where(ELIG).rank(axis=1, pct=True),
}
assert len(VARS) == 24

# Panel largo: una fila por par (protocolo, semana) elegible
ii, jj = np.nonzero(ELIG.to_numpy())


def tomar(M):
    return M.reindex(index=ELIG.index, columns=ELIG.columns).to_numpy(dtype=float)[ii, jj]


columnas = {"semana": ELIG.index[ii], "protocolo": ELIG.columns[jj]}
columnas["categoria"] = info["categoria"].reindex(columnas["protocolo"]).to_numpy()
columnas["tvl"] = tomar(TVL)
for nombre, M in VARS.items():
    columnas[nombre] = tomar(M)
columnas["g_objetivo"] = tomar(G_OBJ)
columnas["y"] = tomar(Y.astype(float)).astype(int)
panel = pd.DataFrame(columnas)
print(f"Panel: {len(panel):,} observaciones de {panel['protocolo'].nunique():,} protocolos en {panel['semana'].nunique()} semanas")
panel.head()

#### B.7 Imputación y codificación (dataset para el Entregable II)
Según el reporte: (i) *fees* y volumen no reportados = 0 más las indicadoras `has_fees` y `has_dex_vol`; (iii) ventanas sin historial suficiente = mediana transversal de la semana; recorte de las variables continuas en los percentiles 1 y 99 de cada semana; categoría en *one-hot*, agrupando en "Otros" las poco frecuentes.

In [ ]:
modelo = panel[["semana", "protocolo", "categoria", "y"]].copy()
modelo["has_fees"] = panel["log_fees_week"].notna().astype(int).to_numpy()
modelo["has_dex_vol"] = panel["log_dex_vol_week"].notna().astype(int).to_numpy()

F0, V0 = FEES.fillna(0), VOL.fillna(0)
LF0, LV0 = np.log1p(F0), np.log1p(V0)
imputadas = {
    "log_fees_week": LF0, "g_fees_1w": LF0 - LF0.shift(1), "g_fees_4w": LF0 - LF0.shift(4),
    "log_dex_vol_week": LV0, "ratio_fees_tvl": F0 / TVL, "ratio_vol_tvl": V0 / TVL,
    "fees_tvl_rank_w": (F0 / TVL).where(ELIG).rank(axis=1, pct=True),
}
for nombre in VARS:
    modelo[nombre] = tomar(imputadas[nombre]) if nombre in imputadas else panel[nombre].to_numpy()

for nombre in VARS:
    if modelo[nombre].isna().any():
        mediana = modelo.groupby("semana")[nombre].transform("median")
        modelo[nombre] = modelo[nombre].fillna(mediana).fillna(modelo[nombre].median())

CONTINUAS = [v for v in VARS if not v.endswith("_rank_w")]
grupos = modelo.groupby("semana")
for v in CONTINUAS:
    bajo = grupos[v].transform(lambda s: s.quantile(0.01))
    alto = grupos[v].transform(lambda s: s.quantile(0.99))
    modelo[v] = modelo[v].clip(bajo, alto)

conteo_cat = modelo.groupby("categoria")["protocolo"].nunique()
frecuentes = conteo_cat[conteo_cat >= MIN_PROTOS_CATEGORIA].index
modelo["categoria_agrupada"] = modelo["categoria"].where(modelo["categoria"].isin(frecuentes), "Otros")
modelo = pd.concat([modelo, pd.get_dummies(modelo["categoria_agrupada"], prefix="cat", dtype=int)], axis=1)
COLS_CAT = [c for c in modelo.columns if c.startswith("cat_")]

panel.to_csv(os.path.join(CARPETA_DATOS, "panel_eda.csv.gz"), index=False)
modelo.to_csv(os.path.join(CARPETA_DATOS, "dataset_modelo.csv.gz"), index=False)
print(f"Dataset del modelo: {modelo.shape[0]:,} filas | 24 variables + 2 indicadoras + {len(COLS_CAT)} columnas de categoría")
print("Categorías:", ", ".join(c[4:] for c in COLS_CAT))

#### B.8 Análisis exploratorio: cifras
Estas cifras alimentan el resumen, la Sec. I-B y la Tabla II del reporte. Los faltantes se cuentan **antes** de imputar.

In [ ]:
N, P, T = len(panel), panel["protocolo"].nunique(), panel["semana"].nunique()
por_semana = panel.groupby("semana").size()
pct_pos = 100 * panel["y"].mean()
pos_semana = 100 * panel.groupby("semana")["y"].mean()
q90_sem = Q90.reindex(por_semana.index)
g = panel["g_objetivo"]

print(f"Observaciones N = {N:,} | protocolos P = {P:,} | semanas T = {T}")
print(f"Protocolos elegibles por semana: mín {por_semana.min()}, mediana {por_semana.median():.0f}, máx {por_semana.max()}")
print(f"Clase positiva: {pct_pos:.1f} % (por semana entre {pos_semana.min():.1f} % y {pos_semana.max():.1f} %)")
print(f"Sin TVL en t+1 (etiqueta 0 por abandono): {100 * g.isna().mean():.2f} %")
print(f"Umbral q0.9: promedio {q90_sem.mean():.4f}, mín {q90_sem.min():.4f}, máx {q90_sem.max():.4f}")
print(f"g_objetivo: mediana {g.median():.4f}, curtosis en exceso {g.kurt():.1f}")

TABLA_II = {"g": g, "gtvl1": panel["g_tvl_1w"], "logfees": panel["log_fees_week"],
            "logdex": panel["log_dex_vol_week"], "ratiofees": panel["ratio_fees_tvl"], "vol12": panel["vol_12w"]}
estad = pd.DataFrame({k: {"media": s.mean(), "mediana": s.median(), "desv": s.std(), "falt": 100 * s.isna().mean()}
                      for k, s in TABLA_II.items()}).T
print("\nTabla II (antes de imputar):")
print(estad.round(4))

faltantes = (100 * panel[list(VARS)].isna().mean()).round(1)
print("\n% de faltantes por variable:")
print(faltantes.to_string())

# Correlación de Spearman entre las 24 variables (pares completos, antes de imputar)
CORR = panel[list(VARS)].corr(method="spearman")
c = CORR.to_numpy()
iu = np.triu_indices(len(VARS), k=1)
pares = pd.DataFrame({"var1": CORR.index[iu[0]], "var2": CORR.columns[iu[1]], "rho": c[iu]}).dropna()
pares = pares.loc[pares["rho"].abs().sort_values(ascending=False).index]
print("\nPares más correlacionados:")
print(pares.head(12).round(3).to_string(index=False))

# Dependencias lineales exactas entre variables (filas sin faltantes)
completas = panel[list(VARS)].dropna()
identidades = {}


def verifica(nombre, a, b):
    if len(completas):
        d = np.nanmax(np.abs(a.to_numpy() - b.to_numpy()))
        identidades[nombre] = bool(d <= 1e-8)


for k in VENTANAS_MOM:
    verifica(f"mom_{k}w = g_tvl_{k}w / {k}", completas[f"mom_{k}w"], completas[f"g_tvl_{k}w"] / k)
verifica("accel = 2 g_tvl_1w - g_tvl_2w", completas["accel"], 2 * completas["g_tvl_1w"] - completas["g_tvl_2w"])
verifica("accel_12w = mom_4w - mom_12w", completas["accel_12w"], completas["mom_4w"] - completas["mom_12w"])
X = completas.loc[:, completas.std() > 0].to_numpy(dtype=float)
RANGO = int(np.linalg.matrix_rank((X - X.mean(0)) / X.std(0))) if len(X) > X.shape[1] else None
print(f"\nFilas completas: {len(completas):,} | rango de la matriz de predictores: {RANGO} de {len(VARS)}")
for nombre, ok in identidades.items():
    print(("✓ " if ok else "✗ ") + nombre)

# Verificación de las cifras de TVL agregado citadas en la Sec. I-A
if not os.path.exists(os.path.join(CARPETA_DATOS, "tvl_agregado.csv")):
    raise RuntimeError("Falta el TVL agregado: vuelva a ejecutar la celda B.3.")
agg = pd.read_csv(os.path.join(CARPETA_DATOS, "tvl_agregado.csv"))
TVLA = pd.Series(pd.to_numeric(agg["tvl"], errors="coerce").to_numpy(), index=a_fechas(agg["date"]))
TVLA = TVLA[TVLA.index.notna() & (TVLA.index <= ULTIMA_SEMANA)].groupby(level=0).last().resample("W-SUN").last()
TVLA = TVLA[TVLA.index >= PRIMERA_SEMANA]
fin_2022 = TVLA[:"2022-12-31"]
max_2025 = TVLA["2025-01-01":"2025-12-31"]
print("\nTVL agregado (para verificar la Sec. I-A):")
if len(fin_2022):
    print(f"  fin de 2022: {fin_2022.iloc[-1] / 1e6:,.0f} M USD")
if len(max_2025):
    print(f"  máximo de 2025: {max_2025.max() / 1e6:,.0f} M USD ({max_2025.idxmax().date()})")
en_2026 = TVLA["2026-01-01":]
if len(en_2026):
    caida = (en_2026 / TVLA.cummax().reindex(en_2026.index) - 1).min()
    print(f"  mayor caída en 2026 desde el máximo previo: {100 * caida:.0f} %")

#### B.9 Análisis exploratorio: figuras
Se guardan en `figs/` en PDF (para el reporte) y PNG (vista previa), con el ancho de una columna IEEE.

In [ ]:
TINTA, TINTA2, TENUE, REJILLA, EJE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
AZUL, NARANJA = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 8, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.edgecolor": EJE, "axes.linewidth": 0.6, "axes.labelcolor": TINTA2,
    "xtick.color": TENUE, "ytick.color": TENUE, "text.color": TINTA,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": REJILLA, "grid.linewidth": 0.5,
    "pdf.fonttype": 42, "figure.dpi": 150, "savefig.dpi": 200,
})
ANCHO = 3.4   # pulgadas: una columna IEEE
ANCHO_MAPA = 2.75   # el mapa de calor, con etiquetas y barra de color, queda de ~3.5 pulgadas


def guardar(fig, nombre):
    fig.savefig(os.path.join(CARPETA_FIGS, nombre + ".pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(CARPETA_FIGS, nombre + ".png"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


def eje_anios(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))


# Fig. 1a: TVL agregado
fig, ax = plt.subplots(figsize=(ANCHO, 1.7))
ax.plot(TVLA.index, TVLA / 1e9, color=AZUL, lw=1.4, solid_joinstyle="round", solid_capstyle="round")
ax.set_xlabel("Semana")
ax.set_ylabel("TVL agregado\n(miles de millones de USD)")
ax.set_ylim(bottom=0)
eje_anios(ax)
guardar(fig, "tvl_agregado")

# Fig. 1b: protocolos elegibles por semana
fig, ax = plt.subplots(figsize=(ANCHO, 1.7))
ax.plot(por_semana.index, por_semana.to_numpy(), color=AZUL, lw=1.4, solid_joinstyle="round", solid_capstyle="round")
ax.set_xlabel("Semana")
ax.set_ylabel("Protocolos elegibles")
ax.set_ylim(bottom=0)
eje_anios(ax)
guardar(fig, "protocolos_elegibles")

# Fig. 2a: distribución de la variable objetivo continua (se omite el 1 % más extremo)
validos = g.dropna()
lo, hi = validos.quantile(0.005), validos.quantile(0.995)
fig, ax = plt.subplots(figsize=(ANCHO, 1.9))
ax.hist(validos[(validos >= lo) & (validos <= hi)], bins=80, color=AZUL, edgecolor="white", linewidth=0.3)
ax.set_yscale("log")
q_media = q90_sem.mean()
ax.axvline(q_media, color=NARANJA, lw=1.2)
ax.annotate(f"$q_{{0.9}}$ promedio = {q_media:.3f}", xy=(q_media, 1), xycoords=("data", "axes fraction"),
            xytext=(4, -2), textcoords="offset points", ha="left", va="top", fontsize=7, color=TINTA2)
ax.set_xlabel("Crecimiento logarítmico semanal del TVL, $g_{i,t+1}$")
ax.set_ylabel("Observaciones (escala log.)")
guardar(fig, "distribucion_crecimiento")
print(f"Se omitió el 1 % de valores más extremos (fuera de [{lo:.2f}, {hi:.2f}]).")

# Fig. 2b: correlación de Spearman entre las 24 variables
divergente = LinearSegmentedColormap.from_list(
    "azul_rojo", ["#104281", "#3987e5", "#9ec5f4", "#f0efec", "#f3b3b2", "#e34948", "#a32b2a"])
fig, ax = plt.subplots(figsize=(ANCHO_MAPA, ANCHO_MAPA))
im = ax.imshow(CORR.to_numpy(), cmap=divergente, vmin=-1, vmax=1)
ax.set_xticks(range(len(VARS)), labels=list(VARS), rotation=90, fontsize=6)
ax.set_yticks(range(len(VARS)), labels=list(VARS), fontsize=6)
ax.grid(False)
for borde in ax.spines.values():
    borde.set_visible(False)
ax.set_xlabel("Variable")
ax.set_ylabel("Variable")
barra = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
barra.set_label("ρ de Spearman", fontsize=7)
barra.ax.tick_params(labelsize=6)
guardar(fig, "correlacion")

### C. Paradigma de aprendizaje
Aprendizaje **supervisado**: clasificación binaria de $y_{i,t}$, que vale 1 si el crecimiento logarítmico del TVL de $t$ a $t+1$ está en el decil superior de la semana (ec. 2 del reporte). La métrica principal es la PR-AUC, porque la clase positiva es cercana al 10 % y un clasificador aleatorio obtiene una PR-AUC igual a esa prevalencia. La validación es temporal: entrenamiento 2022–2024, validación 2025 y prueba 2026, descartando la última semana de los dos primeros bloques porque su etiqueta usa el TVL de la semana siguiente, que ya pertenece al bloque posterior.

La celda siguiente muestra el tamaño y el balance de clases de cada bloque.

In [ ]:
semanas_panel = pd.Series(sorted(panel["semana"].unique()))


def semanas_de(anio_ini, anio_fin, quitar_ultima=True):
    s = semanas_panel[(semanas_panel.dt.year >= anio_ini) & (semanas_panel.dt.year <= anio_fin)]
    return set(s.iloc[:-1]) if (quitar_ultima and len(s)) else set(s)


BLOQUES = {
    "Entrenamiento (2022-2024)": semanas_de(2022, 2024),
    "Validación (2025)": semanas_de(2025, 2025),
    "Prueba (2026)": semanas_de(2026, 2026, quitar_ultima=False),
}
filas = []
for nombre, sem in BLOQUES.items():
    sub = panel[panel["semana"].isin(sem)]
    filas.append({"bloque": nombre, "semanas": len(sem), "observaciones": len(sub),
                  "protocolos": sub["protocolo"].nunique(),
                  "% clase positiva": round(100 * sub["y"].mean(), 1) if len(sub) else float("nan")})
TABLA_BLOQUES = pd.DataFrame(filas).set_index("bloque")
TABLA_BLOQUES

## 2. Estado del arte
Resumen de la Tabla III del reporte. Las métricas que no se vieron en clase (R²oos, NSE, TI, IA, DA y la razón de Sharpe) se definen en la Sec. II-B del reporte.

| Trabajo | Paradigma | Técnica y datos | Validación | Métricas | Resultados |
|---|---|---|---|---|---|
| Ali (2026), *J. Risk Financial Manag.* | Regresión (valoración) | *Super Learner* (Extra Trees, CatBoost y SVR; KNN); 30 protocolos DeFi, datos de DeFiLlama | Validación cruzada de 10 pliegues; validación cruzada temporal de panel | RMSE, MAE, R² | RMSE = 0.085, R² = 0.97 |
| Ghosh et al. (2023), *Int. Rev. Financ. Anal.* | Regresión (precio diario) | ISOMAP + Gradient Boosting y UMAP + Random Forest; 4 tokens DeFi y 4 NFT | Partición 85/15; GridSearchCV | NSE, TI, IA, DA | NSE de 0.85 a 0.98; DA de 0.89 a 0.96 |
| Gu et al. (2020), *Rev. Financ. Stud.* | Regresión (retorno mensual) | De OLS a redes neuronales; ~30 000 acciones y 920 predictores | Entrenamiento 1957–1974, validación 1975–1986, prueba 1987–2016 | R²oos, Sharpe | R²oos de 0.33 a 0.40 %; Sharpe de 1.35 |
| Jaquart et al. (2022), *J. Finance Data Sci.* | Clasificación binaria (retorno ≥ mediana del día siguiente) | LSTM, GRU, TCN, GBC, RF y regresión logística; 100 criptomonedas | 5 ventanas móviles: 500/150/150 días | Exactitud, prueba binomial, Sharpe | Exactitud de 52.9 a 54.2 %; Sharpe de 3.23 |

## Anexo: archivos para el reporte (Overleaf)
`resultados_eda.tex` define cada cifra con `\setdato{clave}{valor}`; `main.tex` la usa con `\dato{clave}`. También se arma un borrador del párrafo de hallazgos: **revíselo** y, si quiere cambiarlo, edite el texto directamente en `resultados_eda.tex` o en `main.tex`.

In [ ]:
def miles(n):
    return f"{int(round(float(n))):,}".replace(",", "\\,")


def num(x, cifras=3):
    """Número con 3 cifras significativas, en modo matemático (signo menos correcto)."""
    if x is None or not np.isfinite(x):
        return "--"
    if x == 0:
        return "$0$"
    dec = max(0, min(6, cifras - 1 - int(np.floor(np.log10(abs(x))))))
    s = f"{x:,.{dec}f}".replace(",", "\\,")
    return f"${s}$"


def fecha(d):
    return pd.Timestamp(d).strftime("%d/%m/%Y")


datos = {
    "fechadescarga": fecha(FECHA_DESCARGA), "ultimasemana": fecha(ULTIMA_SEMANA),
    "N": miles(N), "P": miles(P), "T": miles(T), "p": f"{pct_pos:.1f}",
    "minE": miles(por_semana.min()), "maxE": miles(por_semana.max()), "medE": miles(por_semana.median()),
    "m": str(MIN_PROTOS_CATEGORIA),
    "ncat": str(len(COLS_CAT)),
}
# Tamaño de los bloques de entrenamiento, validación y prueba (Sec. I-C)
for clave, nombre in [("nentr", "Entrenamiento (2022-2024)"), ("nval", "Validación (2025)"), ("nprueba", "Prueba (2026)")]:
    datos[clave] = miles(TABLA_BLOQUES.loc[nombre, "observaciones"])
# Cifras de TVL agregado citadas en la Sec. I-A (redondeadas al millar de M USD)
if len(fin_2022):
    datos["tvl2022"] = miles(round(fin_2022.iloc[-1] / 1e6, -3))
if len(max_2025):
    datos["tvl2025"] = miles(round(max_2025.max() / 1e6, -3))
if len(en_2026):
    datos["caida2026"] = f"{abs(100 * caida):.0f}"

for clave, fila in estad.iterrows():
    for est in ("media", "mediana", "desv"):
        datos[f"t-{clave}-{est}"] = num(fila[est])
    datos[f"t-{clave}-falt"] = f"{fila['falt']:.1f}"

# Borrador del párrafo de hallazgos (Sec. I-B, "Análisis exploratorio")
frases = [
    f"La variable objetivo continua $g_{{i,t+1}}$ tiene mediana {num(g.median())} y colas pesadas "
    f"(curtosis en exceso de {num(g.kurt())}); el umbral semanal $q_{{0.9}}$ promedia {num(q90_sem.mean())} "
    f"y varía entre {num(q90_sem.min())} y {num(q90_sem.max())}, mientras que la clase positiva se mantiene "
    f"entre {pos_semana.min():.1f}\\,\\% y {pos_semana.max():.1f}\\,\\% por semana.",
    f"Las variables de actividad concentran los faltantes: los \\emph{{fees}} faltan en "
    f"{faltantes['log_fees_week']:.1f}\\,\\% de las observaciones y el volumen DEX en "
    f"{faltantes['log_dex_vol_week']:.1f}\\,\\%, lo que justifica las indicadoras de disponibilidad.",
]
redundantes = [n for n, ok in identidades.items() if ok]
if redundantes and RANGO is not None and RANGO < len(VARS):
    frases.append(
        "La correlación de Spearman (Fig.~\\ref{fig:corr}) muestra variables redundantes: cada "
        "\\texttt{mom\\_}$k$\\texttt{w} es igual a \\texttt{g\\_tvl\\_}$k$\\texttt{w}$/k$, y \\texttt{accel} y "
        "\\texttt{accel\\_12w} son "
        f"combinaciones lineales de otras variables, por lo que la matriz de predictores tiene rango {RANGO} de "
        f"{len(VARS)}; estas variables son candidatas a eliminarse en el Entregable~II.")
otros = pares[pares["rho"].abs() < 0.999]
if len(otros):
    a, b, r = otros.iloc[0]
    frases.append(
        f"Entre variables distintas, el par más correlacionado es \\texttt{{{a.replace('_', chr(92) + '_')}}} y "
        f"\\texttt{{{b.replace('_', chr(92) + '_')}}} ($\\rho = {r:.2f}$).")
datos["hallazgos"] = " ".join(frases)

with open("resultados_eda.tex", "w", encoding="utf-8") as f:
    f.write(f"% Generado por EDA_Entregable1.ipynb con datos descargados el {FECHA_DESCARGA.date()}.\n")
    f.write("% Súbalo a Overleaf junto a main.tex. Cada línea define una cifra que main.tex usa con \\dato{clave}.\n")
    for k, v in datos.items():
        f.write(f"\\setdato{{{k}}}{{{v}}}\n")

FIGURAS = ["tvl_agregado", "protocolos_elegibles", "distribucion_crecimiento", "correlacion"]
with zipfile.ZipFile("entrega_overleaf.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("resultados_eda.tex")
    for nombre in FIGURAS:
        z.write(os.path.join(CARPETA_FIGS, nombre + ".pdf"))

print("Borrador del párrafo de hallazgos:\n")
print(datos["hallazgos"])
try:
    from google.colab import files
    files.download("entrega_overleaf.zip")
except ImportError:
    print("\nListo: entrega_overleaf.zip (súbalo a Overleaf).")